<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/17-online-bandits-reinforcement-learning.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Online Learning, Bandits, and Reinforcement Learning**

Most earlier chapters assume that a fixed training set exists before the model is fitted. A **sequential learner** instead predicts or acts while data are still arriving. Its decisions may determine which feedback becomes visible, alter later states, or expose people and systems to real consequences. The central question is therefore broader than “which model fits the data?”:

> What information is available before a decision, what feedback is revealed afterward, and can the current action change future opportunities?

These three questions separate closely related formulations:

| Formulation | Information before action | Feedback afterward | Does action affect the next context or state? | Main objective |
|---|---|---|---|---|
| Batch supervised learning | A fixed labeled dataset | No online feedback is required | No | Generalization risk |
| Online learning | Current example or loss context | Full target or full loss information | Usually treated as exogenous | Regret against a comparator |
| Multi-armed bandit | No context, or only time/history | Reward of the chosen action only | No long-term state effect | Bandit regret |
| Contextual bandit | Current context $x_t$ | Reward of the chosen action only | Context is treated as exogenous | Contextual policy value or regret |
| Reinforcement learning | Current state or observation | Reward and next state | Yes | Expected long-term return |

<div class="diagram-scroll">

![A spectrum from batch prediction to reinforcement learning.](assets/sequential-feedback-spectrum.svg){fig-alt="Batch learning, online prediction, bandits, contextual bandits, and Markov decision processes are arranged by increasingly partial feedback and stronger effects of actions on future states."}

</div>

Using a more powerful formulation is not automatically better. Reinforcement learning adds delayed credit assignment, exploration risk, non-independent data, and difficult evaluation. If an action does not affect future opportunities, a contextual bandit is usually easier and more statistically efficient. If every action's loss is revealed, full-information online learning should exploit that extra feedback.

### **The Online Learning Protocol**

Online learning studies a sequence of rounds $t=1,\ldots,T$. A common supervised protocol is:

1. the learner observes $x_t$ and chooses parameters or a prediction using $w_t$;
2. the outcome $y_t$ or loss function $\ell_t$ is revealed;
3. the learner incurs $\ell_t(w_t)$ and updates to $w_{t+1}$.

The prediction must be made **before** the current outcome is seen. Updating a batch model after every record is computationally incremental, but it is not a valid online evaluation if each prediction was computed after observing its own target.

<div class="diagram-scroll">

![Observe, decide, reveal feedback, and update in the online protocol.](assets/online-learning-protocol.svg){fig-alt="Each online round observes context, makes a prediction before the outcome is revealed, incurs loss, and updates parameters for the next round."}

</div>

Online learning does not require independent and identically distributed observations. The sequence may be stochastic, adversarial, seasonal, or drifting. That flexibility changes the performance criterion from estimating one population risk to comparing cumulative decisions with a clearly defined reference.

#### **Sequential Prediction and Regret**

For a comparator class $\mathcal W$, **static regret** is

$$
R_T
=
\sum_{t=1}^{T}\ell_t(w_t)
-
\min_{w\in\mathcal W}
\sum_{t=1}^{T}\ell_t(w).
$$

The comparator is one fixed decision selected with hindsight. A learner is **no-regret** when $R_T/T\to 0$: its average excess loss vanishes relative to the best fixed comparator, even though the learner had to act sequentially.

This does not imply that the learner follows a changing optimum. **Dynamic regret**

$$
R_T^{\mathrm{dyn}}
=
\sum_{t=1}^{T}\ell_t(w_t)
-
\sum_{t=1}^{T}\ell_t(w_t^\star)
$$

uses a time-varying comparator and is meaningful only after constraining its variation, for example through a path length $\sum_t\lVert w_t^\star-w_{t-1}^\star\rVert$. Without that restriction, the comparator can change arbitrarily after every observation and no learner can compete.

<div class="diagram-scroll">

![Static and dynamic regret use different comparators.](assets/online-regret-comparators.svg){fig-alt="The online learner is compared either with one fixed hindsight decision for static regret or with a constrained time-varying sequence for dynamic regret."}

</div>

Regret is an operational comparison, not a test-set accuracy. It depends on the horizon, loss, comparator class, feedback protocol, and assumptions about the sequence. Report all five.


#### **Online Gradient Descent and the Online Perceptron**

For convex losses over a feasible set $\mathcal W$, **projected online gradient descent (OGD)** performs

$$
w_{t+1}
=
\Pi_{\mathcal W}
\left(
w_t-\eta_t\nabla\ell_t(w_t)
\right),
$$

where $\Pi_{\mathcal W}$ projects an unconstrained update back into the feasible set. Projection is not cosmetic: regret bounds require control of the decision diameter, while practical constraints may encode bounded weights, budgets, or valid probabilities.

If $\mathcal W$ has diameter $D$, every gradient norm is at most $G$, and $\eta=D/(G\sqrt T)$, the standard convex analysis gives

$$
R_T\le DG\sqrt T,
\qquad
\frac{R_T}{T}\le\frac{DG}{\sqrt T}.
$$

The proof expands $\lVert w_{t+1}-w\rVert^2$, uses non-expansiveness of projection and convexity, then telescopes distances across rounds. The same update resembles stochastic gradient descent, but the interpretation differs: SGD estimates an expected batch objective under sampling assumptions, whereas OGD controls sequential regret even for a non-random loss sequence.

<details>
<summary><strong>Python: run projected OGD and compute realized static regret</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(17)
horizon, dimensions = 2_000, 5

# Bound every context norm so the loss gradients cannot grow without control.
contexts = rng.normal(size=(horizon, dimensions))
contexts /= np.maximum(1.0, np.linalg.norm(contexts, axis=1, keepdims=True))
true_weight = np.array([1.5, -1.0, 0.7, 0.2, -0.4])
targets = contexts @ true_weight + rng.normal(scale=0.12, size=horizon)

radius = 3.0
weight = np.zeros(dimensions)
online_losses = []

for time_index, (context, target) in enumerate(zip(contexts, targets), start=1):
    prediction = context @ weight
    error = prediction - target
    online_losses.append(0.5 * error**2)

    # The current target is used only after the current prediction is scored.
    gradient = error * context
    learning_rate = 0.8 / np.sqrt(time_index)
    candidate = weight - learning_rate * gradient

    # Euclidean projection onto ||w||_2 <= radius.
    candidate_norm = np.linalg.norm(candidate)
    weight = candidate * min(1.0, radius / max(candidate_norm, 1e-12))

# The static comparator may inspect the full sequence, but must use one weight.
comparator, *_ = np.linalg.lstsq(contexts, targets, rcond=None)
comparator *= min(1.0, radius / max(np.linalg.norm(comparator), 1e-12))
comparator_losses = 0.5 * (contexts @ comparator - targets) ** 2

static_regret = np.sum(online_losses) - np.sum(comparator_losses)
print("online cumulative loss:", round(float(np.sum(online_losses)), 3))
print("best fixed cumulative loss:", round(float(np.sum(comparator_losses)), 3))
print("static regret:", round(float(static_regret), 3))
print("average regret per round:", round(float(static_regret / horizon), 5))
print("distance to comparator:", round(float(np.linalg.norm(weight - comparator)), 3))
```

</details>

For binary classification, the **online Perceptron** predicts $\hat y_t=\operatorname{sign}(w_t^\top x_t)$ and updates only after a mistake:

$$
w_{t+1}
=
w_t+y_tx_t
\quad\text{if }y_tw_t^\top x_t\le 0.
$$

Suppose there is a unit vector $u$ with margin $y_tu^\top x_t\ge\gamma>0$ and $\lVert x_t\rVert\le R$. The classic mistake bound is

$$
M\le\left(\frac{R}{\gamma}\right)^2.
$$

Each mistake increases alignment $u^\top w$ by at least $\gamma$, while the squared norm grows by at most $R^2$. Combining both facts limits how many mistakes are possible. The bound is deterministic and sequence-order independent, but it collapses when the data are not separable; margin losses, regularization, averaging, or probabilistic online models are then more suitable.

<details>
<summary><strong>Python: verify the Perceptron update on a margin-separated stream</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(23)
separator = np.array([0.8, -0.5, 0.3])
separator /= np.linalg.norm(separator)

examples, labels = [], []
while len(examples) < 1_000:
    point = rng.normal(size=3)
    point /= max(1.0, np.linalg.norm(point))
    signed_distance = point @ separator

    # Reject points close to the boundary to construct a positive margin.
    if abs(signed_distance) >= 0.25:
        examples.append(point)
        labels.append(1 if signed_distance > 0 else -1)

examples = np.asarray(examples)
labels = np.asarray(labels)
order = rng.permutation(len(labels))

weight = np.zeros(3)
mistakes = 0
for index in order:
    if labels[index] * (weight @ examples[index]) <= 0:
        weight += labels[index] * examples[index]
        mistakes += 1

margin = np.min(labels * (examples @ separator))
max_radius = np.max(np.linalg.norm(examples, axis=1))
mistake_bound = (max_radius / margin) ** 2

predictions = np.where(examples @ weight >= 0, 1, -1)
print("observed mistakes:", mistakes)
print("theoretical upper bound:", round(float(mistake_bound), 1))
print("stream training accuracy:", round(float(np.mean(predictions == labels)), 3))
print("cosine with true separator:", round(float(weight @ separator / np.linalg.norm(weight)), 3))
```

</details>

Online updates react quickly but can also chase noise. Learning-rate schedules, forgetting factors, sliding windows, change-point alarms, adaptive regret, and explicit reset policies should be chosen according to how rapidly the environment can change. Always use **prequential evaluation**: predict first, record the loss, then learn from that observation.


### **Multi-Armed Bandits**

A stochastic $K$-armed bandit has unknown reward distributions $\nu_1,\ldots,\nu_K$ with means $\mu_1,\ldots,\mu_K$. At round $t$, the learner chooses arm $A_t$ and observes only its reward $R_t(A_t)$. Rewards for unchosen arms are counterfactual and remain hidden.

If $\mu^\star=\max_a\mu_a$, expected pseudo-regret is

$$
\bar R_T
=
T\mu^\star
-
\mathbb E\left[\sum_{t=1}^{T}\mu_{A_t}\right]
=
\sum_{a=1}^{K}\Delta_a\,\mathbb E[N_a(T)],
$$

where $\Delta_a=\mu^\star-\mu_a$ and $N_a(T)$ counts pulls of arm $a$. The decomposition shows that regret comes from selecting suboptimal arms, but some suboptimal pulls are necessary to identify the best arm.

<div class="diagram-scroll">

![Full information and bandit feedback reveal different outcomes.](assets/bandit-feedback-exploration.svg){fig-alt="Full-information learning observes every action loss, while a bandit observes only the selected reward and must deliberately explore hidden alternatives."}

</div>

Bandit models apply to one-step interventions such as selecting one recommendation, notification, advert, or treatment when the current decision does not alter a modeled future state. Delayed rewards, interference between users, inventory constraints, and repeated exposure can violate this reduction.

#### **Exploration versus Exploitation**

**Exploitation** chooses the action with the best current estimate. **Exploration** chooses an uncertain or apparently inferior action because its outcome may improve future decisions. Pure greed can lock onto the first arm that receives lucky rewards; uniform exploration continues wasting trials after uncertainty has already been resolved.

Good exploration depends on the uncertainty model:

- stochastic stationary bandits support concentration bounds or Bayesian posteriors;
- adversarial bandits require algorithms such as EXP3 rather than stochastic UCB;
- drifting rewards require discounting, windows, restarts, or change detection;
- safety constraints may forbid unrestricted exploration even when it reduces statistical regret.

The horizon also matters. Exploration has value only if enough future decisions remain to recover its immediate opportunity cost.

#### **Epsilon-Greedy, UCB, and Thompson Sampling**

**Epsilon-greedy** chooses the empirically best arm with probability $1-\epsilon$ and explores uniformly with probability $\epsilon$. It is transparent, but its exploration is not directed toward uncertainty. A constant $\epsilon$ produces linear regret asymptotically; a decaying schedule can improve regret but may adapt poorly after drift.

For bounded stochastic rewards, **UCB1** selects

$$
A_t
=
\arg\max_a
\left[
\hat\mu_a(t)
+
\sqrt{\frac{2\log t}{N_a(t)}}
\right].
$$

The first term exploits estimated reward; the confidence bonus is large for rarely sampled arms. This is **optimism under uncertainty**: behave as if each plausible arm might be as good as its upper confidence bound.

For Bernoulli rewards, **Thompson sampling** maintains

$$
\mu_a\mid\mathcal H_t
\sim
\operatorname{Beta}(\alpha_a,\beta_a),
$$

draws $\tilde\mu_a$ from every posterior, and chooses $\arg\max_a\tilde\mu_a$. A plausible arm is selected approximately in proportion to its posterior probability of being optimal.

<div class="diagram-scroll">

![Epsilon-greedy, UCB, and Thompson sampling represent uncertainty differently.](assets/bandit-algorithm-beliefs.svg){fig-alt="Epsilon-greedy uses random exploration, UCB adds a confidence bonus, and Thompson sampling chooses the arm whose posterior draw is largest."}

</div>

<details>
<summary><strong>Python: compare epsilon-greedy, UCB, and Thompson sampling</strong></summary>

```python
import numpy as np

arm_means = np.array([0.25, 0.35, 0.50, 0.55])
horizon = 2_000
repetitions = 60


def run_bandit(algorithm, seed):
    rng = np.random.default_rng(seed)
    counts = np.zeros(len(arm_means), dtype=int)
    successes = np.zeros(len(arm_means))
    chosen_arms = np.empty(horizon, dtype=int)

    for time_index in range(horizon):
        if algorithm == "epsilon-greedy":
            empirical_means = successes / np.maximum(counts, 1)
            if rng.random() < 0.10 or np.any(counts == 0):
                available = np.flatnonzero(counts == 0)
                arm = int(rng.choice(available if len(available) else len(arm_means)))
            else:
                arm = int(np.argmax(empirical_means))

        elif algorithm == "ucb":
            if np.any(counts == 0):
                arm = int(np.flatnonzero(counts == 0)[0])
            else:
                empirical_means = successes / counts
                bonus = np.sqrt(2.0 * np.log(time_index + 1) / counts)
                arm = int(np.argmax(empirical_means + bonus))

        elif algorithm == "thompson":
            samples = rng.beta(1.0 + successes, 1.0 + counts - successes)
            arm = int(np.argmax(samples))
        else:
            raise ValueError("unknown algorithm")

        reward = rng.binomial(1, arm_means[arm])
        counts[arm] += 1
        successes[arm] += reward
        chosen_arms[time_index] = arm

    pseudo_regret = np.sum(np.max(arm_means) - arm_means[chosen_arms])
    late_optimal_rate = np.mean(chosen_arms[-300:] == np.argmax(arm_means))
    return pseudo_regret, late_optimal_rate


for algorithm in ["epsilon-greedy", "ucb", "thompson"]:
    results = np.array(
        [run_bandit(algorithm, seed=10_000 + run) for run in range(repetitions)]
    )
    print(
        f"{algorithm:15s}",
        "mean regret =", round(float(results[:, 0].mean()), 1),
        "late optimal-arm rate =", round(float(results[:, 1].mean()), 3),
    )
```

</details>

One simulation is not an algorithm ranking. Change the gaps, horizon, reward family, prior, drift, and tuning parameters. Report regret distributions across independent runs, not only the best seed.

#### **Contextual Bandits**

A contextual bandit observes $x_t$ before acting and learns a policy $\pi(a\mid x)$. Under a linear reward model,

$$
\mathbb E[R_t(a)\mid x_t]
=
x_t^\top\theta_a,
$$

**LinUCB** maintains a ridge-regression estimate for each action and chooses

$$
A_t
=
\arg\max_a
\left[
x_t^\top\hat\theta_a
+
\alpha\sqrt{x_t^\top A_a^{-1}x_t}
\right].
$$

The uncertainty bonus is context-specific: an action can be well understood for one type of user and uncertain for another.

<div class="diagram-scroll">

![The contextual bandit protocol observes context before choosing one action.](assets/contextual-bandit-protocol.svg){fig-alt="Context enters a policy, one action is selected, only that action's reward is observed, and the selected action model is updated."}

</div>

<details>
<summary><strong>Python: implement LinUCB and compare it with a context-free UCB</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(31)
horizon, dimensions, actions = 2_500, 4, 3
true_parameters = np.array(
    [
        [0.9, -0.6, 0.2, 0.0],
        [-0.5, 0.8, 0.0, 0.4],
        [0.1, -0.2, 0.9, -0.7],
    ]
)

contexts = rng.normal(size=(horizon, dimensions))
expected_rewards = contexts @ true_parameters.T


def run_linucb(alpha=0.8):
    covariance = np.repeat(np.eye(dimensions)[None, :, :], actions, axis=0)
    reward_sum = np.zeros((actions, dimensions))
    selected = []

    for context, means in zip(contexts, expected_rewards):
        scores = []
        for action in range(actions):
            inverse = np.linalg.inv(covariance[action])
            estimate = inverse @ reward_sum[action]
            bonus = alpha * np.sqrt(context @ inverse @ context)
            scores.append(context @ estimate + bonus)

        action = int(np.argmax(scores))
        reward = means[action] + rng.normal(scale=0.25)
        covariance[action] += np.outer(context, context)
        reward_sum[action] += reward * context
        selected.append(action)

    selected = np.asarray(selected)
    regret = np.sum(np.max(expected_rewards, axis=1) - expected_rewards[np.arange(horizon), selected])
    optimal_rate = np.mean(selected == np.argmax(expected_rewards, axis=1))
    return regret, optimal_rate


def run_context_free_ucb():
    counts = np.zeros(actions, dtype=int)
    reward_totals = np.zeros(actions)
    selected = []

    for time_index, means in enumerate(expected_rewards):
        if np.any(counts == 0):
            action = int(np.flatnonzero(counts == 0)[0])
        else:
            scores = reward_totals / counts + np.sqrt(
                2.0 * np.log(time_index + 1) / counts
            )
            action = int(np.argmax(scores))

        reward = means[action] + rng.normal(scale=0.25)
        counts[action] += 1
        reward_totals[action] += reward
        selected.append(action)

    selected = np.asarray(selected)
    regret = np.sum(np.max(expected_rewards, axis=1) - expected_rewards[np.arange(horizon), selected])
    optimal_rate = np.mean(selected == np.argmax(expected_rewards, axis=1))
    return regret, optimal_rate


linucb_regret, linucb_rate = run_linucb()
ucb_regret, ucb_rate = run_context_free_ucb()
print("LinUCB regret / optimal rate:", round(float(linucb_regret), 1), round(float(linucb_rate), 3))
print("context-free UCB:", round(float(ucb_regret), 1), round(float(ucb_rate), 3))
```

</details>

In deployed contextual bandits, log the context, available action set, selected action, reward definition, delay, policy version, and **selection propensity**. Without action probabilities and adequate overlap, unbiased off-policy evaluation may be impossible. Contexts that are affected by earlier actions also break the one-step abstraction and may require an MDP.


### **Markov Decision Processes**

A **Markov decision process (MDP)** models sequential decisions whose actions influence future states. A discounted MDP is commonly written

$$
\mathcal M=(\mathcal S,\mathcal A,P,R,\gamma,\rho_0),
$$

where $\mathcal S$ is the state space, $\mathcal A$ the action space, $P(s'\mid s,a)$ the transition kernel, $R$ the reward model, $\gamma\in[0,1)$ the discount factor, and $\rho_0$ the initial-state distribution.

The Markov assumption is

$$
P(S_{t+1},R_{t+1}\mid S_0,A_0,\ldots,S_t,A_t)
=
P(S_{t+1},R_{t+1}\mid S_t,A_t).
$$

It does not say the world has no history. It says the chosen state representation contains all history needed to predict the next transition and reward. If hidden information matters, the observation is not a Markov state; recurrent models, belief states, or partially observable MDPs may be required.

#### **States, Actions, Rewards, and Transitions**

- A **state** should support prediction and control, not merely describe what is easy to log.
- An **action** is a decision the learner can actually intervene on.
- A **reward** specifies immediate preference, not a label describing the correct action.
- A **transition** specifies how the action changes future state probabilities.
- A **terminal state** ends an episode; truncation caused by a time limit is conceptually different and may still require bootstrapping.

<div class="diagram-scroll">

![A small MDP with safe and risky transitions.](assets/mdp-transition-graph.svg){fig-alt="A start state branches through safe and risky actions toward a terminal goal with different transition probabilities, illustrating that actions change future states."}

</div>

The agent-environment interface makes the time ordering explicit:

<div class="diagram-scroll">

![Agent-environment interaction in reinforcement learning.](assets/rl-agent-environment.svg){fig-alt="The agent sends an action to the environment and receives the next state and reward in return."}

</div>

*Image source: [Wikimedia Commons, Agent-environment-diagram-rl.svg](https://commons.wikimedia.org/wiki/File:Agent-environment-diagram-rl.svg), released under CC0.*

Indexing conventions differ across books and libraries. One common convention is

$$
S_t \xrightarrow{A_t} (R_{t+1},S_{t+1}).
$$

State the convention once, then use it consistently. Many implementation errors are one-step indexing errors disguised as algorithmic failures.

#### **Policies, Returns, and Discounting**

A stochastic policy is

$$
\pi(a\mid s)=P(A_t=a\mid S_t=s).
$$

It maps states to action distributions; it is not the same object as a value function. A trajectory

$$
\tau=(S_0,A_0,R_1,S_1,A_1,R_2,\ldots)
$$

is generated jointly by the initial distribution, policy, and environment transitions.

The discounted return from time $t$ is

$$
G_t
=
\sum_{k=0}^{\infty}\gamma^kR_{t+k+1}
=
R_{t+1}+\gamma G_{t+1}.
$$

Discounting ensures finite values in continuing tasks, expresses time preference or uncertainty, and controls the effective horizon, approximately $1/(1-\gamma)$. It should not be used to hide an incorrectly specified episode. In finite-horizon problems, time may need to be part of the state because the optimal action can change as the deadline approaches.

<details>
<summary><strong>Python: simulate trajectories and discounted returns in a chain MDP</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(41)
terminal_state = 4
discount = 0.95


def step(state, action):
    # Action 0 intends left; action 1 intends right.
    intended_direction = -1 if action == 0 else 1
    actual_direction = intended_direction if rng.random() < 0.85 else -intended_direction
    next_state = int(np.clip(state + actual_direction, 0, terminal_state))
    reward = 1.0 if next_state == terminal_state else -0.02
    return next_state, reward, next_state == terminal_state


def sample_episode(max_steps=100):
    state = 0
    rewards = []
    states = [state]

    for _ in range(max_steps):
        # A fixed stochastic policy chooses right with probability 0.8.
        action = int(rng.random() < 0.8)
        state, reward, terminated = step(state, action)
        states.append(state)
        rewards.append(reward)
        if terminated:
            break

    discounted_return = sum(
        (discount**time_index) * reward
        for time_index, reward in enumerate(rewards)
    )
    return states, rewards, discounted_return


episodes = [sample_episode() for _ in range(5_000)]
returns = np.array([episode[2] for episode in episodes])
lengths = np.array([len(episode[1]) for episode in episodes])

print("example states:", episodes[0][0])
print("example rewards:", episodes[0][1])
print("mean episode length:", round(float(lengths.mean()), 2))
print("mean discounted return:", round(float(returns.mean()), 3))
print("return standard deviation:", round(float(returns.std(ddof=1)), 3))
```

</details>

Reward design is part of system specification. A proxy reward can create shortcut behavior, unsafe exploration, or optimization that looks successful while violating the real objective. Track constraints and outcome metrics separately instead of compressing every concern into one scalar reward.


### **Bellman Equations**

Bellman equations express a long-horizon value through one immediate transition plus the value of what follows. They are recursive consistency equations, not separate assumptions added to the MDP.

#### **State and Action Value Functions**

For policy $\pi$, the state-value and action-value functions are

$$
V^\pi(s)
=
\mathbb E_\pi[G_t\mid S_t=s],
$$

$$
Q^\pi(s,a)
=
\mathbb E_\pi[G_t\mid S_t=s,A_t=a].
$$

The Bellman expectation equations are

$$
V^\pi(s)
=
\sum_a\pi(a\mid s)
\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma V^\pi(s')\right],
$$

$$
Q^\pi(s,a)
=
\sum_{s',r}p(s',r\mid s,a)
\left[
r+\gamma\sum_{a'}\pi(a'\mid s')Q^\pi(s',a')
\right].
$$

The optimal functions satisfy

$$
V^\star(s)
=
\max_a
\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma V^\star(s')\right],
$$

$$
Q^\star(s,a)
=
\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma\max_{a'}Q^\star(s',a')\right].
$$

Expectation equations evaluate a specified policy; optimality equations include a maximization and are nonlinear. Confusing the two changes the problem from “how good is this policy?” to “what is the best policy?”

<div class="diagram-scroll">

![A Bellman backup combines immediate reward and next-state value.](assets/bellman-backup.svg){fig-alt="A current value is updated through one-step reward and discounted next value, using an expectation for policy evaluation or a maximum for optimal control."}

</div>

For a finite MDP and fixed policy,

$$
V^\pi=r_\pi+\gamma P_\pi V^\pi,
\qquad
V^\pi=(I-\gamma P_\pi)^{-1}r_\pi.
$$

Direct solution is useful for small problems and validation; iterative methods are preferable when the state space is large or sparse.

<details>
<summary><strong>Python: solve Bellman expectation equations for a stochastic policy</strong></summary>

```python
import numpy as np

states, actions = 5, 2
terminal = 4
discount = 0.95
transition = np.zeros((states, actions, states))
reward = np.zeros((states, actions, states))

for state in range(states - 1):
    for action in range(actions):
        intended = -1 if action == 0 else 1
        for probability, direction in [(0.85, intended), (0.15, -intended)]:
            next_state = int(np.clip(state + direction, 0, terminal))
            transition[state, action, next_state] += probability
            reward[state, action, next_state] = (
                1.0 if next_state == terminal else -0.02
            )

# Terminal states transition to themselves without additional reward.
transition[terminal, :, terminal] = 1.0
policy = np.tile([0.2, 0.8], (states, 1))

policy_transition = np.einsum("sa,san->sn", policy, transition)
state_action_reward = np.sum(transition * reward, axis=2)
policy_reward = np.sum(policy * state_action_reward, axis=1)

value = np.linalg.solve(
    np.eye(states) - discount * policy_transition,
    policy_reward,
)
action_value = state_action_reward + discount * np.einsum(
    "san,n->sa", transition, value
)

print("V^pi:", np.round(value, 3))
print("Q^pi at start [left, right]:", np.round(action_value[0], 3))
print("policy consistency at start:", round(float(policy[0] @ action_value[0]), 3))
```

</details>

A Bellman residual measures local inconsistency,

$$
\delta(s)
=
r_\pi(s)+\gamma(P_\pi V)(s)-V(s).
$$

A small residual under the evaluated model is a useful numerical check, but it does not protect against a wrong transition model, reward misspecification, hidden state, or deployment shift.


### **Dynamic Programming**

Dynamic programming (DP) assumes the transition and reward model is known and that expected backups can be computed. It is therefore a **planning** method. Reinforcement learning uses sampled experience when that model is unavailable, but many model-free algorithms can be understood as stochastic approximations to DP backups.

#### **Policy Evaluation and Policy Iteration**

**Iterative policy evaluation** repeatedly applies the Bellman expectation operator:

$$
V_{k+1}(s)
\leftarrow
\sum_a\pi(a\mid s)
\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma V_k(s')\right].
$$

For $\gamma<1$, this operator is a contraction in the maximum norm and converges to the unique $V^\pi$.

The policy improvement theorem replaces the current policy with an action greedy with respect to $Q^\pi$:

$$
\pi'(s)
\in
\arg\max_a Q^\pi(s,a).
$$

Alternating evaluation and improvement yields **policy iteration**. Evaluation may be exact or stopped early; modified policy iteration spans the space between full policy iteration and value iteration.

<div class="diagram-scroll">

![Policy iteration alternates evaluation and improvement.](assets/dynamic-programming-loop.svg){fig-alt="Policy evaluation computes the current policy value, policy improvement makes the policy greedy, and the loop repeats until the policy is stable."}

</div>

<details>
<summary><strong>Pseudocode: policy iteration</strong></summary>

```text
initialize policy pi
repeat:
    evaluate V^pi from Bellman expectation backups
    stable = true
    for each nonterminal state s:
        old_action = pi(s)
        pi(s) = argmax_a sum_{s',r} p(s',r|s,a)[r + gamma V^pi(s')]
        stable = stable and (pi(s) == old_action)
until stable
```

</details>

#### **Value Iteration**

Value iteration applies the Bellman optimality operator directly:

$$
V_{k+1}(s)
\leftarrow
\max_a
\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma V_k(s')\right].
$$

Once values converge, a greedy policy is extracted. The stopping tolerance controls value error; an unchanged printed policy is not by itself proof that the values have converged.

<details>
<summary><strong>Python: implement policy iteration and value iteration on a grid</strong></summary>

```python
import numpy as np

rows, columns = 4, 4
terminal = rows * columns - 1
discount = 0.95
directions = [(-1, 0), (0, 1), (1, 0), (0, -1)]
symbols = np.array(["U", "R", "D", "L"])


def next_state(state, action):
    if state == terminal:
        return terminal
    row, column = divmod(state, columns)
    row_shift, column_shift = directions[action]
    next_row = int(np.clip(row + row_shift, 0, rows - 1))
    next_column = int(np.clip(column + column_shift, 0, columns - 1))
    return next_row * columns + next_column


def evaluate_policy(policy, tolerance=1e-10):
    value = np.zeros(rows * columns)
    sweeps = 0
    while True:
        updated = value.copy()
        for state in range(rows * columns):
            if state != terminal:
                successor = next_state(state, policy[state])
                updated[state] = -1.0 + discount * value[successor]
        sweeps += 1
        if np.max(np.abs(updated - value)) < tolerance:
            return updated, sweeps
        value = updated


def improve_policy(value):
    policy = np.zeros(rows * columns, dtype=int)
    for state in range(rows * columns):
        if state != terminal:
            action_values = [
                -1.0 + discount * value[next_state(state, action)]
                for action in range(4)
            ]
            policy[state] = int(np.argmax(action_values))
    return policy


def policy_iteration():
    policy = np.zeros(rows * columns, dtype=int)
    improvement_steps = 0
    evaluation_sweeps = 0
    while True:
        value, sweeps = evaluate_policy(policy)
        evaluation_sweeps += sweeps
        improved = improve_policy(value)
        improvement_steps += 1
        if np.array_equal(improved, policy):
            return policy, value, improvement_steps, evaluation_sweeps
        policy = improved


def value_iteration(tolerance=1e-10):
    value = np.zeros(rows * columns)
    sweeps = 0
    while True:
        updated = value.copy()
        for state in range(rows * columns):
            if state != terminal:
                updated[state] = max(
                    -1.0 + discount * value[next_state(state, action)]
                    for action in range(4)
                )
        sweeps += 1
        if np.max(np.abs(updated - value)) < tolerance:
            return improve_policy(updated), updated, sweeps
        value = updated


pi_policy, pi_value, improvement_steps, evaluation_sweeps = policy_iteration()
vi_policy, vi_value, vi_sweeps = value_iteration()

def show_policy(policy):
    labels = symbols[policy].astype(object)
    labels[terminal] = "G"
    return labels.reshape(rows, columns)

print("policy iteration improvements / evaluation sweeps:", improvement_steps, evaluation_sweeps)
print(show_policy(pi_policy))
print("value iteration sweeps:", vi_sweeps)
print(show_policy(vi_policy))
print("maximum value difference:", round(float(np.max(np.abs(pi_value - vi_value))), 10))
```

</details>

Synchronous backups use values from the previous sweep; in-place or prioritized updates often converge faster but change the update order. For large state spaces, exact enumeration becomes impossible, motivating sampling, function approximation, simulation models, and approximate planning.


### **Model-Free Reinforcement Learning**

When $P$ and $R$ are unknown, an agent can estimate values from sampled transitions. **Model-free** means the algorithm does not learn or use an explicit transition model for planning; it does not mean the environment has no structure or that assumptions disappear.

#### **Monte Carlo and Temporal-Difference Learning**

Monte Carlo (MC) policy evaluation waits until a return $G_t$ is available and updates

$$
V(S_t)
\leftarrow
V(S_t)
+
\alpha\left[G_t-V(S_t)\right].
$$

It does not bootstrap, but episodic returns can have high variance and updates are delayed until termination.

One-step temporal-difference learning, TD(0), updates after each transition:

$$
\delta_t
=
R_{t+1}+\gamma V(S_{t+1})-V(S_t),
$$

$$
V(S_t)
\leftarrow
V(S_t)+\alpha\delta_t.
$$

TD uses a sampled reward and next state, then bootstraps from its current next-state estimate. Its target is biased while $V$ is inaccurate, but it usually has lower variance and works naturally in continuing tasks.

<div class="diagram-scroll">

![Monte Carlo, TD, and dynamic programming construct different value targets.](assets/mc-td-backups.svg){fig-alt="Monte Carlo waits for a sampled full return, TD uses one sampled transition and a bootstrap value, while dynamic programming computes an expected backup from a known model."}

</div>

The distinction is **sampling versus expectation** and **bootstrapping versus complete return**. Multi-step TD methods and eligibility traces interpolate between MC and one-step TD.

<details>
<summary><strong>Python: compare Monte Carlo and TD(0) on the random walk</strong></summary>

```python
import numpy as np

nonterminal_states = np.arange(1, 6)
true_value = nonterminal_states / 6.0
episodes = 200
runs = 100
checkpoints = [1, 10, 50, 200]


def generate_episode(rng):
    state = 3
    trajectory = []
    while state not in (0, 6):
        next_state = state + (-1 if rng.random() < 0.5 else 1)
        reward = 1.0 if next_state == 6 else 0.0
        trajectory.append((state, reward, next_state))
        state = next_state
    return trajectory


def learn(method, seed, alpha=0.10):
    rng = np.random.default_rng(seed)
    value = np.full(7, 0.5)
    value[[0, 6]] = [0.0, 0.0]
    errors = []

    for episode_index in range(1, episodes + 1):
        trajectory = generate_episode(rng)

        if method == "mc":
            returns = np.zeros(len(trajectory))
            running_return = 0.0
            for index in range(len(trajectory) - 1, -1, -1):
                running_return = trajectory[index][1] + running_return
                returns[index] = running_return

            visited = set()
            for index, (state, _, _) in enumerate(trajectory):
                if state not in visited:  # first-visit MC
                    value[state] += alpha * (returns[index] - value[state])
                    visited.add(state)

        elif method == "td":
            for state, reward, next_state in trajectory:
                bootstrap = 0.0 if next_state in (0, 6) else value[next_state]
                target = reward + bootstrap
                value[state] += alpha * (target - value[state])
        else:
            raise ValueError("unknown method")

        errors.append(np.sqrt(np.mean((value[1:6] - true_value) ** 2)))

    return np.asarray(errors)


for method in ["mc", "td"]:
    error_curves = np.array(
        [learn(method, seed=50_000 + run) for run in range(runs)]
    )
    summary = {
        checkpoint: round(float(error_curves[:, checkpoint - 1].mean()), 3)
        for checkpoint in checkpoints
    }
    print(method.upper(), "mean RMSE by episode:", summary)
```

</details>

No single method dominates every problem. MC is attractive when episodes are short and complete returns are trustworthy. TD is attractive for long or continuing trajectories and online updates. Compare learning curves over independent seeds and measure sensitivity to $\alpha$, initialization, episode truncation, and reward scale.

#### **SARSA and Q-Learning**

For control, the value of state-action pairs must be learned while the policy changes.

**SARSA** is on-policy:

$$
Q(S_t,A_t)
\leftarrow
Q(S_t,A_t)
+
\alpha
\left[
R_{t+1}
+\gamma Q(S_{t+1},A_{t+1})
-Q(S_t,A_t)
\right].
$$

The target uses the next action actually selected by the behavior policy. With persistent $\epsilon$-greedy exploration, it values the consequences of exploratory actions.

**Q-learning** is off-policy:

$$
Q(S_t,A_t)
\leftarrow
Q(S_t,A_t)
+
\alpha
\left[
R_{t+1}
+\gamma\max_aQ(S_{t+1},a)
-Q(S_t,A_t)
\right].
$$

Its behavior may explore, but its target acts greedily. In the tabular setting with sufficient exploration and suitable decreasing step sizes, Q-learning can converge to $Q^\star$.

<div class="diagram-scroll">

![SARSA and Q-learning can prefer different cliff-walking routes.](assets/sarsa-qlearning-cliff.svg){fig-alt="SARSA learns a safer route because its on-policy target accounts for exploratory cliff falls, whereas Q-learning's greedy target favors the shortest route along the cliff."}

</div>

<details>
<summary><strong>Python: compare SARSA and Q-learning in cliff walking</strong></summary>

```python
import numpy as np

rows, columns, actions = 4, 12, 4
start = (3, 0)
goal = (3, 11)
cliff = {(3, column) for column in range(1, 11)}
directions = [(-1, 0), (0, 1), (1, 0), (0, -1)]


def environment_step(state, action):
    row, column = state
    row_shift, column_shift = directions[action]
    candidate = (
        int(np.clip(row + row_shift, 0, rows - 1)),
        int(np.clip(column + column_shift, 0, columns - 1)),
    )
    if candidate in cliff:
        return start, -100.0, False, True
    if candidate == goal:
        return goal, -1.0, True, False
    return candidate, -1.0, False, False


def epsilon_greedy(q_values, state, rng, epsilon=0.10):
    if rng.random() < epsilon:
        return int(rng.integers(actions))
    row, column = state
    best = np.flatnonzero(q_values[row, column] == q_values[row, column].max())
    return int(rng.choice(best))


def train(method, seed, episodes=300, alpha=0.50, discount=1.0):
    rng = np.random.default_rng(seed)
    q_values = np.zeros((rows, columns, actions))
    episode_returns, cliff_falls = [], []

    for _ in range(episodes):
        state = start
        action = epsilon_greedy(q_values, state, rng)
        total_reward = 0.0
        falls = 0

        for _ in range(1_000):
            next_state, reward, terminated, fell = environment_step(state, action)
            total_reward += reward
            falls += int(fell)

            if terminated:
                target = reward
                next_action = None
            elif method == "sarsa":
                next_action = epsilon_greedy(q_values, next_state, rng)
                target = reward + discount * q_values[next_state][next_action]
            elif method == "q-learning":
                next_action = epsilon_greedy(q_values, next_state, rng)
                target = reward + discount * np.max(q_values[next_state])
            else:
                raise ValueError("unknown method")

            q_values[state][action] += alpha * (
                target - q_values[state][action]
            )
            if terminated:
                break
            state, action = next_state, next_action

        episode_returns.append(total_reward)
        cliff_falls.append(falls)

    return q_values, np.asarray(episode_returns), np.asarray(cliff_falls)


for method in ["sarsa", "q-learning"]:
    outcomes = [
        train(method, seed=70_000 + run)[1:]
        for run in range(8)
    ]
    returns = np.array([outcome[0] for outcome in outcomes])
    falls = np.array([outcome[1] for outcome in outcomes])
    print(
        method,
        "last-30 mean return =", round(float(returns[:, -30:].mean()), 1),
        "cliff falls / episode =", round(float(falls[:, -30:].mean()), 3),
    )
```

</details>

“On-policy” and “off-policy” describe the relationship between the policy generating data and the policy being evaluated or improved. They do not mean online versus offline, nor do they guarantee safety. Off-policy methods need adequate coverage of target actions; with function approximation, bootstrapping and off-policy updates can become unstable.


### **Function Approximation and Deep RL**

Tabular methods store one value per state or state-action pair. They become infeasible when states are images, continuous measurements, long histories, or combinatorial configurations. Function approximation replaces the table with

$$
\hat V(s;w),
\qquad
\hat Q(s,a;w),
$$

so experience in one state can generalize to related states.

This generalization introduces approximation error and optimization instability. The combination of

1. **function approximation**,
2. **bootstrapping**, and
3. **off-policy learning**

is known as the **deadly triad** because it can make value estimates diverge, even when each ingredient is useful by itself. Neural networks add non-convex optimization, correlated trajectories, shifting data distributions, and extrapolation to poorly covered actions.

#### **DQN as a Bridge to Deep Learning**

The Deep Q-Network (DQN) minimizes a temporal-difference loss

$$
\mathcal L(\theta)
=
\mathbb E_{(s,a,r,s',d)\sim\mathcal B}
\left[
\operatorname{Huber}
\left(
y-Q_\theta(s,a)
\right)
\right],
$$

with target

$$
y
=
r
+
\gamma(1-d)\max_{a'}Q_{\theta^-}(s',a').
$$

Here $d$ marks a true terminal transition, $\mathcal B$ is an experience replay distribution, and $\theta^-$ is a delayed target-network copy.

<div class="diagram-scroll">

![DQN combines an online network, replay buffer, target network, and TD loss.](assets/dqn-stabilizers.svg){fig-alt="Transitions enter a replay buffer, random minibatches train an online Q-network, and a delayed target network supplies more stable bootstrap targets."}

</div>

The main stabilizers serve different purposes:

- **experience replay** reuses data and weakens short-range temporal correlation;
- **target networks** slow movement of the bootstrap target;
- **Huber loss and gradient clipping** reduce sensitivity to large TD errors;
- **reward scaling or clipping** controls numerical range, but changes the objective if performed carelessly;
- **Double DQN** separates action selection from target evaluation to reduce maximization bias.

Replay makes DQN off-policy, but replayed experience still needs action coverage. A neural network can assign arbitrarily high values to actions absent from the data, which is a central challenge in offline RL.

<details>
<summary><strong>Python: train a compact DQN with replay and a target network</strong></summary>

```python
from collections import deque
import random
import numpy as np
import torch
from torch import nn

torch.manual_seed(17)
np.random.seed(17)
random.seed(17)
torch.set_num_threads(1)

state_count, action_count = 7, 2
terminal_state = state_count - 1


def transition(state, action, rng):
    # A short chain with 10% action slip and a goal on the right.
    intended = -1 if action == 0 else 1
    direction = intended if rng.random() > 0.10 else -intended
    next_state = int(np.clip(state + direction, 0, terminal_state))
    terminated = next_state == terminal_state
    reward = 1.0 if terminated else -0.02
    return next_state, reward, terminated


def one_hot(state):
    vector = np.zeros(state_count, dtype=np.float32)
    vector[state] = 1.0
    return vector


def make_network():
    return nn.Sequential(
        nn.Linear(state_count, 32),
        nn.ReLU(),
        nn.Linear(32, action_count),
    )


online_network = make_network()
target_network = make_network()
target_network.load_state_dict(online_network.state_dict())
optimizer = torch.optim.Adam(online_network.parameters(), lr=2e-3)
replay = deque(maxlen=5_000)
rng = np.random.default_rng(17)
optimization_steps = 0

for episode in range(250):
    state = 0
    epsilon = max(0.05, 1.0 - episode / 200)

    for _ in range(40):
        if rng.random() < epsilon:
            action = int(rng.integers(action_count))
        else:
            with torch.no_grad():
                action = int(
                    online_network(torch.tensor(one_hot(state))).argmax()
                )

        next_state, reward, terminated = transition(state, action, rng)
        replay.append((state, action, reward, next_state, terminated))
        state = next_state

        if len(replay) >= 64:
            batch = random.sample(replay, 32)
            states, actions, rewards, next_states, terminals = zip(*batch)
            state_tensor = torch.tensor(
                np.stack([one_hot(item) for item in states])
            )
            next_state_tensor = torch.tensor(
                np.stack([one_hot(item) for item in next_states])
            )
            action_tensor = torch.tensor(actions, dtype=torch.long)
            reward_tensor = torch.tensor(rewards, dtype=torch.float32)
            terminal_tensor = torch.tensor(terminals, dtype=torch.float32)

            predicted_q = online_network(state_tensor).gather(
                1, action_tensor[:, None]
            ).squeeze(1)
            with torch.no_grad():
                next_q = target_network(next_state_tensor).max(dim=1).values
                target_q = reward_tensor + 0.97 * (1.0 - terminal_tensor) * next_q

            loss = nn.functional.smooth_l1_loss(predicted_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(online_network.parameters(), 5.0)
            optimizer.step()
            optimization_steps += 1

            if optimization_steps % 75 == 0:
                target_network.load_state_dict(online_network.state_dict())

        if terminated:
            break


def evaluate(episodes=300):
    evaluation_rng = np.random.default_rng(999)
    returns, successes = [], []
    for _ in range(episodes):
        state, total_reward = 0, 0.0
        for _ in range(40):
            with torch.no_grad():
                action = int(
                    online_network(torch.tensor(one_hot(state))).argmax()
                )
            state, reward, terminated = transition(
                state, action, evaluation_rng
            )
            total_reward += reward
            if terminated:
                break
        returns.append(total_reward)
        successes.append(state == terminal_state)
    return np.asarray(returns), np.asarray(successes)


evaluation_returns, successes = evaluate()
with torch.no_grad():
    start_q = online_network(torch.tensor(one_hot(0))).numpy()
print("greedy success rate:", round(float(successes.mean()), 3))
print("mean undiscounted return:", round(float(evaluation_returns.mean()), 3))
print("Q(start, left/right):", np.round(start_q, 3))
print("replay transitions:", len(replay))
```

</details>

This example is intentionally small enough to audit. Large deep-RL experiments need multiple seeds, complete learning curves, environment and wrapper versions, evaluation without exploration noise, wall-clock and sample budgets, and ablations of replay, target updates, reward processing, and network architecture. A high final score from one seed is not reliable evidence.


### **Policy Optimization**

Value-based methods infer a policy by maximizing an estimated action value. **Policy optimization** parameterizes the policy directly, which is especially useful for continuous actions, stochastic behavior, constrained distributions, or policies whose probability density must be modeled explicitly.

#### **Policy Gradients and Actor-Critic Methods**

Let a differentiable policy $\pi_\theta$ induce trajectories $\tau$. The objective is

$$
J(\theta)
=
\mathbb E_{\tau\sim\pi_\theta}
\left[
\sum_{t=0}^{T-1}\gamma^tR_{t+1}
\right].
$$

The policy-gradient theorem gives the form

$$
\nabla_\theta J(\theta)
\propto
\mathbb E_{\pi_\theta}
\left[
\nabla_\theta\log\pi_\theta(A_t\mid S_t)
Q^{\pi_\theta}(S_t,A_t)
\right].
$$

The log-probability term increases the probability of actions weighted by their estimated long-term quality. **REINFORCE** replaces $Q^\pi$ with a sampled return. The estimator is unbiased under its sampling assumptions but often has high variance.

A baseline $b(s)$ can be subtracted without changing the expected gradient:

$$
\mathbb E
\left[
\nabla_\theta\log\pi_\theta(A_t\mid S_t)
\left(G_t-b(S_t)\right)
\right].
$$

Choosing $b(s)\approx V^\pi(s)$ produces an advantage estimate. An **actor-critic** method learns:

- an **actor** $\pi_\theta(a\mid s)$ that chooses actions;
- a **critic** $V_\phi(s)$ or $Q_\phi(s,a)$ that estimates value;
- an advantage or TD error that trains the actor.

For a one-step critic,

$$
\delta_t
=
R_{t+1}
+\gamma V_\phi(S_{t+1})
-V_\phi(S_t)
$$

acts as an advantage estimate.

<div class="diagram-scroll">

![The actor chooses actions while the critic estimates advantage.](assets/actor-critic-loop.svg){fig-alt="An actor sends actions to the environment, the critic evaluates resulting transitions, and an advantage signal updates the actor."}

</div>

<details>
<summary><strong>Python: train a one-step actor-critic on a chain environment</strong></summary>

```python
import numpy as np
import torch
from torch import nn
from torch.distributions import Categorical

torch.manual_seed(29)
torch.set_num_threads(1)
rng = np.random.default_rng(29)

state_count, action_count = 7, 2
terminal_state = state_count - 1


def encode(state):
    observation = torch.zeros(state_count)
    observation[state] = 1.0
    return observation


def step(state, action):
    intended = -1 if action == 0 else 1
    direction = intended if rng.random() > 0.10 else -intended
    next_state = int(np.clip(state + direction, 0, terminal_state))
    terminated = next_state == terminal_state
    reward = 1.0 if terminated else -0.02
    return next_state, reward, terminated


actor = nn.Sequential(
    nn.Linear(state_count, 24),
    nn.Tanh(),
    nn.Linear(24, action_count),
)
critic = nn.Sequential(
    nn.Linear(state_count, 24),
    nn.Tanh(),
    nn.Linear(24, 1),
)
optimizer = torch.optim.Adam(
    list(actor.parameters()) + list(critic.parameters()),
    lr=3e-3,
)
discount = 0.97

for episode in range(250):
    state = 0
    for _ in range(40):
        observation = encode(state)
        distribution = Categorical(logits=actor(observation))
        action = distribution.sample()
        next_state, reward, terminated = step(state, int(action))

        value = critic(observation).squeeze()
        with torch.no_grad():
            next_value = (
                torch.tensor(0.0)
                if terminated
                else critic(encode(next_state)).squeeze()
            )
            td_target = torch.tensor(reward) + discount * next_value

        advantage = td_target - value
        actor_loss = (
            -distribution.log_prob(action) * advantage.detach()
            - 0.01 * distribution.entropy()
        )
        critic_loss = 0.5 * advantage.pow(2)

        optimizer.zero_grad()
        (actor_loss + critic_loss).backward()
        nn.utils.clip_grad_norm_(
            list(actor.parameters()) + list(critic.parameters()), 5.0
        )
        optimizer.step()

        state = next_state
        if terminated:
            break


def evaluate_policy(episodes=300):
    evaluation_rng = np.random.default_rng(1_029)
    successes, returns = [], []
    for _ in range(episodes):
        state, total_reward = 0, 0.0
        for _ in range(40):
            with torch.no_grad():
                action = int(actor(encode(state)).argmax())
            intended = -1 if action == 0 else 1
            direction = intended if evaluation_rng.random() > 0.10 else -intended
            state = int(np.clip(state + direction, 0, terminal_state))
            terminated = state == terminal_state
            reward = 1.0 if terminated else -0.02
            total_reward += reward
            if terminated:
                break
        successes.append(state == terminal_state)
        returns.append(total_reward)
    return np.mean(successes), np.mean(returns)


success_rate, mean_return = evaluate_policy()
with torch.no_grad():
    start_probabilities = torch.softmax(actor(encode(0)), dim=0).numpy()
    start_value = float(critic(encode(0)))
print("greedy success rate:", round(float(success_rate), 3))
print("mean undiscounted return:", round(float(mean_return), 3))
print("policy at start [left, right]:", np.round(start_probabilities, 3))
print("critic value at start:", round(start_value, 3))
```

</details>

Actor-critic methods introduce coupled approximation: the actor changes the critic's data distribution, while critic error changes the actor update. Entropy bonuses, generalized advantage estimation, trust regions, clipping, replay corrections, and target critics address different failure modes. Their names do not replace diagnostics: inspect returns, episode lengths, entropy, KL divergence, value loss, advantage scale, constraint violations, and seed variation.


### **Exploration, Off-Policy Evaluation, and Safety**

Exploration in an MDP is harder than in a bandit because information may require a sequence of actions, rewards can be delayed, and an unsafe action can move the agent into an irreversible state. Common mechanisms include:

- random action noise or entropy regularization;
- optimism and confidence bonuses;
- count-based or pseudo-count novelty;
- intrinsic rewards based on prediction error or information gain;
- posterior sampling over models or value functions;
- demonstrations, curricula, simulators, or reset mechanisms.

More novelty is not necessarily better. Prediction error can reward stochastic noise, action noise can be meaningless in constrained spaces, and an intrinsic objective can distract from the deployed goal. Evaluate state coverage, task return, constraint violations, and the cost of data collection separately.

When a candidate policy $\pi$ must be evaluated from data generated by a different behavior policy $b$, the problem is **off-policy evaluation (OPE)**. In a contextual bandit with logged tuples $(x_i,a_i,r_i,p_i)$ and known propensity $p_i=b(a_i\mid x_i)$, inverse propensity scoring estimates

$$
\hat V_{\mathrm{IPS}}(\pi)
=
\frac{1}{n}
\sum_{i=1}^{n}
\frac{\pi(a_i\mid x_i)}{b(a_i\mid x_i)}
r_i.
$$

It is unbiased under correct propensities, consistency, and overlap, but may have enormous variance. The self-normalized estimator divides by the sum of weights and trades finite-sample bias for stability.

A direct reward model $\hat q(x,a)$ estimates policy value through

$$
\hat V_{\mathrm{DM}}(\pi)
=
\frac{1}{n}\sum_i\sum_a\pi(a\mid x_i)\hat q(x_i,a).
$$

The doubly robust estimator combines both:

$$
\hat V_{\mathrm{DR}}(\pi)
=
\frac{1}{n}\sum_i
\left[
\sum_a\pi(a\mid x_i)\hat q(x_i,a)
+
\frac{\pi(a_i\mid x_i)}{b(a_i\mid x_i)}
\left(r_i-\hat q(x_i,a_i)\right)
\right].
$$

Cross-fitting reduces overfitting bias when the same log is used to train $\hat q$ and estimate value.

<details>
<summary><strong>Python: compare direct, IPS, self-normalized, and doubly robust OPE</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

rng = np.random.default_rng(53)
sample_count, dimensions, actions = 20_000, 5, 3
contexts = rng.normal(size=(sample_count, dimensions))
parameters = np.array(
    [
        [0.9, -0.5, 0.3, 0.0, 0.2],
        [-0.4, 0.8, 0.1, 0.5, -0.2],
        [0.2, -0.1, 0.9, -0.6, 0.4],
    ]
)


def sigmoid(values):
    return 1.0 / (1.0 + np.exp(-values))


def softmax(values):
    shifted = values - values.max(axis=1, keepdims=True)
    exponentiated = np.exp(shifted)
    return exponentiated / exponentiated.sum(axis=1, keepdims=True)


true_reward_probability = sigmoid(contexts @ parameters.T)

# Both policies retain positive probability for every action.
behavior_base = softmax(1.2 * true_reward_probability + np.array([0.3, 0.0, -0.2]))
behavior_policy = 0.15 / actions + 0.85 * behavior_base
target_base = softmax(5.0 * true_reward_probability)
target_policy = 0.05 / actions + 0.95 * target_base

uniforms = rng.random(sample_count)
logged_actions = (
    uniforms[:, None] > np.cumsum(behavior_policy, axis=1)
).sum(axis=1)
logged_probabilities = behavior_policy[np.arange(sample_count), logged_actions]
logged_rewards = rng.binomial(
    1,
    true_reward_probability[np.arange(sample_count), logged_actions],
)

# Cross-fitted outcome predictions: each row is predicted by a model that
# did not train on that row.
predicted_rewards = np.zeros((sample_count, actions))
folds = KFold(n_splits=2, shuffle=True, random_state=53)
for train_index, validation_index in folds.split(contexts):
    for action in range(actions):
        action_train = train_index[logged_actions[train_index] == action]
        model = LogisticRegression(max_iter=1_000)
        model.fit(contexts[action_train], logged_rewards[action_train])
        predicted_rewards[validation_index, action] = model.predict_proba(
            contexts[validation_index]
        )[:, 1]

target_probability_logged = target_policy[
    np.arange(sample_count), logged_actions
]
weights = target_probability_logged / logged_probabilities
direct_per_row = np.sum(target_policy * predicted_rewards, axis=1)

direct = np.mean(direct_per_row)
ips = np.mean(weights * logged_rewards)
self_normalized = np.sum(weights * logged_rewards) / np.sum(weights)
doubly_robust = np.mean(
    direct_per_row
    + weights
    * (
        logged_rewards
        - predicted_rewards[np.arange(sample_count), logged_actions]
    )
)
true_value = np.mean(
    np.sum(target_policy * true_reward_probability, axis=1)
)
effective_sample_size = np.sum(weights) ** 2 / np.sum(weights**2)

print("true target-policy value:", round(float(true_value), 4))
print("direct / IPS / SNIPS / DR:", np.round(
    [direct, ips, self_normalized, doubly_robust], 4
))
print("99th percentile / max weight:", np.round(
    [np.quantile(weights, 0.99), weights.max()], 2
))
print("effective sample size:", round(float(effective_sample_size), 1))
```

</details>

For multi-step RL, trajectory importance ratios multiply across time and variance can grow exponentially with horizon. Per-decision weighting, fitted value methods, model-based estimators, and doubly robust sequential estimators introduce different bias-variance trade-offs. No estimator can recover actions or states that the behavior policy never visits.

<div class="diagram-scroll">

![Off-policy evaluation and safety checks form deployment gates.](assets/ope-safety-gates.svg){fig-alt="Logged data passes through overlap and uncertainty diagnostics, explicit safety constraints, and a staged deployment gate before a policy is broadly released."}

</div>

Safety should be modeled explicitly. A constrained MDP can optimize

$$
\max_\pi J_R(\pi)
\quad\text{subject to}\quad
J_{C_j}(\pi)\le d_j,\qquad j=1,\ldots,m,
$$

where $C_j$ measures separate costs such as collision, unfair exposure, latency, energy use, or human intervention. A reward penalty is not equivalent to a hard constraint unless its trade-off is justified.

A defensible policy-development process includes:

1. an immutable logging and data-quality specification;
2. overlap and effective-sample-size diagnostics;
3. simulator validation without treating the simulator as ground truth;
4. offline policy evaluation with uncertainty intervals and sensitivity analysis;
5. explicit constraints, overrides, rate limits, and rollback;
6. a small monitored rollout before broader deployment;
7. post-deployment drift, reward, cost, and subgroup monitoring.

Reward hacking, distribution shift, unobserved confounding, delayed effects, and feedback loops are specification problems as much as algorithm problems. Human oversight should have real authority to pause or reverse the policy.


### **Choosing a Sequential Learning Formulation**

Start with the weakest formulation that captures the real consequence of an action:

<div class="diagram-scroll">

![A decision tree for choosing online learning, bandits, or reinforcement learning.](assets/sequential-formulation-tree.svg){fig-alt="If actions affect future states use an MDP; otherwise chosen-action-only feedback suggests a bandit, while full feedback suggests online learning."}

</div>

| Question | If yes | If no |
|---|---|---|
| Are decisions required before all targets are available? | Consider a sequential protocol | Batch learning may be sufficient |
| Does the action alter later states or opportunities? | MDP / reinforcement learning | Continue to feedback visibility |
| Is only the chosen action's reward observed? | Bandit or contextual bandit | Full-information online learning |
| Is context observed before the action? | Contextual policy | Context-free bandit |
| Is a trustworthy transition model available? | Planning or model-based RL | Model-free or learned-model methods |
| Can unsafe actions be explored online? | Still impose constraints and monitoring | Prefer logged data, simulation, demonstrations, or conservative deployment |

Before selecting an algorithm, write down:

1. the unit and timing of state, context, action, reward, and terminal signals;
2. whether actions causally affect future observations;
3. the feedback that is visible for unchosen actions;
4. the comparator, return, horizon, discount, and constraints;
5. the behavior policy and propensities that generate training data;
6. the valid generalization unit: future time, new users, new environments, or new tasks;
7. the baseline policy, deployment gate, monitoring plan, and rollback path.

The main algorithm families can then be compared:

| Method | Requires a model? | Learns on/off policy | Bootstraps? | Main strength | Main risk |
|---|---:|---|---:|---|---|
| OGD / online Perceptron | No | Full-information stream | No | Regret guarantees under weak sequence assumptions | Wrong comparator or slow drift tracking |
| UCB / Thompson sampling | No | Online bandit interaction | No | Efficient one-step exploration | Stationarity and safety assumptions |
| Dynamic programming | Yes | Planning, not sampled behavior | Yes | Exact tabular solution | Model and state-space dependence |
| Monte Carlo | No | Usually on-policy evaluation | No | Complete-return target | High variance and delayed updates |
| SARSA | No | On-policy | Yes | Accounts for exploratory behavior | Sample efficiency and policy dependence |
| Q-learning / DQN | No | Off-policy | Yes | Reuse data and target greedy control | Extrapolation and instability |
| Policy gradient | No | Usually on-policy | Return or critic dependent | Direct stochastic or continuous policy optimization | High variance and sample cost |
| Actor-critic | No | Either, with corrections | Yes | Lower-variance policy learning | Coupled actor/critic error |

A sequential method is successful only when it improves decisions under the **actual feedback and consequence structure**, not when it achieves a high simulator return. Use simple baselines, multiple seeds, learning curves, uncertainty intervals, resource accounting, constraint metrics, and staged deployment.

Primary and official resources include [A Modern Introduction to Online Learning](https://arxiv.org/abs/1912.13213), the free online book [Bandit Algorithms](https://tor-lattimore.com/downloads/book/book.pdf), the original [UCB1 analysis](https://doi.org/10.1023/A:1013689704352), Sutton and Barto's [Reinforcement Learning: An Introduction](https://mitpress.mit.edu/9780262039246/reinforcement-learning/), [Stanford CS234](https://web.stanford.edu/class/cs234/modules.html), [Berkeley CS285](https://rail.eecs.berkeley.edu/deeprlcourse-fa23/), [OpenAI Spinning Up](https://spinningup.openai.com/en/latest/), the original [DQN paper](https://doi.org/10.1038/nature14236), the [policy-gradient theorem paper](https://proceedings.neurips.cc/paper/1999/hash/464d828b85b0bed98e80ade0a5c43b0f-Abstract.html), [doubly robust policy evaluation](https://arxiv.org/abs/1103.4601), and [Constrained Policy Optimization](https://proceedings.mlr.press/v70/achiam17a.html).
